## 🌍 *Tutorial #2*: **Learning Air Pollution Dynamics from Data and Physics**

In the previous tutorial, we tried to answer the question:

> Can the model predict how the pollutant spreads at **later times** and across the **the enitre domain**, where no measurements are available?

For this, we tested a standard **data-driven approach**, but the results did not fully convince our supervisor — we clearly need a better strategy.

### 🧪 **Diffusion Equation**

By discussing the problem with our colleagues, we learned that the evoluation of the concentration $C(x,t)$ should follow a simple **diffusion equation** of the form

$
\begin{equation}
\frac{\partial C}{\partial t} = D \frac{\partial C^2}{\partial^2 x},
\end{equation}
$
where $D$ is a diffusion coefficient. Experimental measurements suggest that, for our substance, it is approximately $D=0.01$.

This is valuable information that we can directly incorporate into our neural network model.

---

### ⚙️ Idea behind Physics-Informed Neural Networks (PINNs)

Instead of relying solely on data, we can use a method called **Physics-Informed Neural Networks (PINNs)**, where we embed known physical laws—such as the diffusion equation—directly into the learning process.

This is achieved by adding an additional physics-based loss term to the training objective. The neural network is then trained not only to fit the observed data, but also to satisfy the governing differential equation.

In this way, the model is penalized whenever its predictions violate known physical laws, leading to solutions that are both data-consistent and physically meaningful.

### 🚀 Next step

Before implementing the PINN, we first load the measuremnt data from the last tutorial. We have provided a predefined function that does this for us and provides the training-ready datasets. We will skip the visualization step here, since it was already covered in the previous tutorial, and instead focus directly on setting up the training data for the physics-informed model.

In [ ]:
from methods.data import load_measurements
X_train, y_train, X_test, y_test = load_measurements(test_size=0.2)

---

## 🧠 Implementing the PINN

Now that the data is prepared, we move to the core idea of this tutorial: incorporating **physical knowledge directly into the neural network training process**.

Unlike the purely data-driven model, a PINN does not only learn from observations. Instead, it also enforces that the prediction satisfies the governing physical law—in our case, the **diffusion equation**.

### ⚙️ Key idea

We assume the neural network approximates the solution:

$
\begin{equation}
C(x,t)\approx C_\theta(x,t)
\end{equation}
$
where $\theta$ represents the trainable parameters of the network.

To enforce physics, we compute derivatives of the network output using **automatic differentiation** and define a second loss term based on the residual of the diffusion equation:

$
\begin{equation}
\mathcal{R}(x,t)=\frac{\partial C_\theta}{\partial t} - D \frac{\partial C^2_\theta}{\partial^2 x}
\end{equation}
$
If the network perfectly satisfies the physics, this residual becomes zero.

### 📉 PINN loss function

The total loss consists of two parts:

- **Data loss**: ensures agreement with observed measurements
- **Physics loss**: enforces the diffusion equation
$
\begin{equation}
\mathcal{L} = \mathcal{L}_\mathrm{data} + \lambda\mathcal{L}_\mathrm{physics}
\end{equation}
$

where:

- $\lambda$ controls the **balance between data and physics** - we can think of the physics loss acting as a regularization,
- $\mathcal{L}_\mathrm{physics}=\frac{1}{N_\mathrm{col}}\sum_{i=1}^{N_\mathrm{col}}\|\mathcal{R}(x^{(i)},t^{(i)})\|^2$ - the **mean squared error of physics residuals** evaluated on what we call **collocation points** $\{(x^{(i)},t^{(i)})\}_{i=1}^{N_\mathrm{col}}$

### 🧩 What changes compared to the previous model?

The network architecture remains the same as before. The key difference is:

- we now additionally train on **collocation points** sampled across the full domain $(x,t)$,
- we use **automatic differentiation** to compute derivatives,
- and we include the **physics residual in the loss function**.

### 🚀 Next step

We will now implement:

1. Sampling of collocation points
2. Computation of derivatives using PyTorch autograd
3. The combined physics-informed loss function

This will allow us to compare this approach with the previous data-driven model, and observe the impact of incorporating physical knowledge into the model.

## 📌 Step 1: Define collocation points (physics points)

These are points (**coordinates in the computantional domain**) where we enforce the physics constraint. The use of collocation points for the physics loss function is sometimes referred to as an *semi-supervised* approach, as we do not need any labels (concentration values) for the computation - just the $x$ and $t$ coordinates where we want to enfore the physics equations within the domain.

In [ ]:
import numpy as np
import torch

def sample_collocation_points(N_col=128):
    """
    Samples collocation points within the computational domain.

    Parameters
    ----------
    N_col : int
        Number of collocation points to be sampled

    Returns
    -------
    X_col : torch.tensor
        Sampled collocation points
    """


    # Sample uniformly in the full domain
    x_col = np.random.uniform(0.0, 1.0, N_col)
    t_col = np.random.uniform(0.0, 1.0, N_col)

    X_col = torch.tensor(
        np.column_stack([x_col, t_col]),
        dtype=torch.float32,
        requires_grad=True
    )
    return X_col

## 🧠 Step 2: Physics residual (diffusion equation)

We now compute:

$
\begin{equation}
\frac{\partial C}{\partial t} = D \frac{\partial C^2}{\partial^2 x},
\end{equation}
$

using automatic differentiation. For this purpose, the collocation points are passed through the network in a forward pass. Using automatic differentiation, we then compute the **derivatives of the network output with respect to its inputs**, i.e. $\frac{\partial C}{\partial t}$ and $\frac{\partial C^2}{\partial^2 x}$ is evaluated at the collocation points. These quantities then help us to formulate the physics residuals as we defined them above.

In [ ]:
def compute_pde_residual(model, X_col, D=0.01):
    """
    Compute the physics residual of the one-dimensional diffusion equation
    at a set of collocation points.

    Parameters
    ----------
    model : torch.nn.Module
        Neural network approximating C(x,t).

    X_col : torch.Tensor
        Tensor of collocation points with columns [x, t].

    D : float, default=0.01
        Diffusion coefficient.

    Returns
    -------
    residuals : torch.Tensor
        PDE residual evaluated at all collocation points.
    """

    x = X_col[:, 0:1]
    t = X_col[:, 1:2]

    # forward pass
    C = model(torch.cat([x, t], dim=1))

    # First-order derivatives
    C_t = torch.autograd.grad(
        C, t,
        grad_outputs=torch.ones_like(C),
        retain_graph=True,
        create_graph=True
    )[0]

    C_x = torch.autograd.grad(
        C, x,
        grad_outputs=torch.ones_like(C),
        retain_graph=True,
        create_graph=True
    )[0]

    # Second-order derivative
    C_xx = torch.autograd.grad(
        C_x, x,
        grad_outputs=torch.ones_like(C_x),
        retain_graph=True,
        create_graph=True
    )[0]

    # Diffusion residual
    residual = C_t - D * C_xx

    return residual

## 🔁 Step 3: PINN loss function and Training Loop

We now combine:

- Data loss (supervised learning)
- Physics loss (PDE constraint)

through 
$
\begin{equation}
\mathcal{L} = \mathcal{L}_\mathrm{data} + \lambda\mathcal{L}_\mathrm{physics}
\end{equation}
$

and use $\lambda$ to trade between both objectives. Optimizing both losses can be considered as a **multi-objective optimization problem**, where $\lambda$ helps us to balance both objectives. 

In [ ]:
def MO_loss(model, X_data, y_data, X_col, lambda_phys=1.0):
    """
    Compute the multi-objective training loss used for the PINN.

    Parameters
    ----------
    model : torch.nn.Module
        Neural network approximating the concentration field C(x,t).

    X_data : torch.Tensor
        Input coordinates with known target values.

    y_data : torch.Tensor
        Target concentration values corresponding to X_data.

    X_col : torch.Tensor
        Collocation points where the PDE residual is evaluated.

    lambda_phys : float, default=1.0
        Weighting factor for the physics loss term.

    Returns
    -------
    loss : torch.Tensor
        Total weighted loss.

    loss_data : torch.Tensor
        Mean squared error on labeled data points.

    loss_phys : torch.Tensor
        Mean squared PDE residual at collocation points.
    """
    # -------------------------
    # Data loss
    # -------------------------
    C_pred = model(X_data)
    loss_data = torch.mean((C_pred - y_data) ** 2)

    # -------------------------
    # Physics loss
    # -------------------------
    residual = compute_pde_residual(model, X_col)
    loss_phys = torch.mean(residual ** 2)

    # Total loss / multi-objective loss
    loss = loss_data + lambda_phys * loss_phys

    return loss, loss_data, loss_phys


def train_pinn(model, optimizer,
               X_data, y_data,
               X_test, y_test,
               N_col=1000,
               n_epochs=4000,
               lambda_phys=1.0,
               print_every=100):

    history = {
        "epoch": [],
        "loss": [],
        "loss_data": [],
        "loss_phys": [],
        "loss_test": []
    }

    for epoch in range(1, n_epochs + 1):

        # -------------------------
        # Training step
        # -------------------------
        model.train()
        optimizer.zero_grad()

        # Collocation points are sampled anew at each epoch
        X_col = sample_collocation_points(N_col=N_col)

        # Multi-obejctive loss is evaluated
        loss, loss_d, loss_p = MO_loss(
            model, X_data, y_data, X_col, lambda_phys
        )

        loss.backward()
        optimizer.step()

        # -------------------------
        # Evaluation step
        # -------------------------
        model.eval()

        with torch.no_grad():
            y_pred_test = model(X_test)
            loss_test = torch.mean((y_pred_test - y_test) ** 2)

        # -------------------------
        # Log history and print current losses
        # -------------------------
        history["epoch"].append(epoch)
        history["loss"].append(loss.item())
        history["loss_data"].append(loss_d.item())
        history["loss_phys"].append(loss_p.item())
        history["loss_test"].append(loss_test.item())

        if epoch % print_every == 0 or epoch == 1:
            print(
                f"Epoch {epoch:5d} | "
                f"Total: {loss.item():1.2e} | "
                f"Data: {loss_d.item():1.2e} | "
                f"PDE: {loss_p.item():1.2e} | "
                f"Test Loss = {loss_test.item():1.2e}"
            )

    return history

---

## ▶️ PINN Training

Having implemented all required components, it is now time to train and evaluate the PINN. We use for the basic model the same network architecture as in the previous tutorial - we have provided a predefined class we can directly load 

In [ ]:
from methods.model import NeuralNet
from torch.optim import Adam

model = NeuralNet()
optimizer = Adam(model.parameters(), lr=1e-3)

N_col = 128
n_epochs = 4000
lambda_phys = 1.0

history = train_pinn(
    model=model,
    optimizer=optimizer,
    X_data=X_train,
    y_data=y_train,
    X_test=X_test,
    y_test=y_test,
    N_col=N_col,
    n_epochs=n_epochs,
    lambda_phys=lambda_phys,
    print_every=100
)

One difference compared to the purely data-driven model may already be apparent: **training takes longer**.

This can be explained by the **additional collocation points** that are included during optimization. Not only does passing these points through the network increase the computational cost, but more importantly, the use of automatic differentiation to compute the required derivatives is typically the main bottleneck of physics-informed training. For this reason, the number of collocation points should be chosen carefully.

However, the expectation is that this **additional computational effort leads to better predictive performance**. At this stage, we therefore accept the increased training cost in exchange for improved results.

It is also worth noting that this overhead only affects the training phase. During inference—that is, when generating predictions—the PINN requires essentially the same amount of time as the purely data-driven model, since the physics loss and derivative computations are no longer needed.

Now let us have a look at the learning curves.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))

plt.plot(history["epoch"], history["loss_data"], label="Data Loss")
plt.plot(history["epoch"], history["loss_phys"], label="Physics Loss")
plt.plot(history["epoch"], history["loss_test"], label="Test Loss")

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training History")
plt.legend()
plt.show()

Although we may initially observe somewhat unusual behavior of the physics loss, all loss terms should decrease toward the end of training, which is a good sign.

This early training phase is quite typical for PINNs. At network initialization, the predictions (concentration values) are often centered around zero. The **zero function is a trivial solution** to many differential equations and therefore can produce physics residuals close to zero at the beginning of training.

As the network starts adapting to the observed data and moves toward a solution that better reflects the true dynamics, the physics loss may temporarily increase. This happens because the model leaves the trivial zero solution and explores more realistic concentration fields. At the same time, the decrease in the data and test losses indicates that the optimization is progressing in the right direction.

---

## 🔬 Comparison to Reference Data

Now let's compare the PINN prediction with the true reference data we obtained initially and already used in the data-driven comparison. We have wrapped the reference plot into a function we can directly use for this.

In [ ]:
from methods.plots import plot_reference
plot_reference(model)

Well, this looks much better compared to the purely data-driven method—we achieved a relative $L_2$ error below $15\%$. And we obtained this improvement **without using any additional observations**, but simply by incorporating the differential equation that governs the physical spreading of the concentration.

You are encouraged to revisit the training process and experiment with the additional design choices that PINNs provide:

- the number of **collocation points**,
- and the parameter $\lambda$, which controls the **balance between fitting the data and satisfying the physics**.

For example:

- if you set $\lambda=0$, the model effectively reduces to the standard data-driven neural network, and you should observe performance similar to that in the previous tutorial,
- reducing the number of collocation points weakens the influence of the physics loss, but may speed up training.

### 📌 A few additional remarks on PINNs

In this tutorial, we used the PINN in a type of **mixed forward/inverse problem setting**. That means we assumed the governing equation was known and combined it with noisy measurement data to infer the pollutant evolution from it.

From a physical perspective, however, some information is still missing that would make the solution **uniquely determined**. In classical PDE forward problems, this additional information is usually provided through:

- **Initial conditions** (e.g. concentration at $t=0$),
- **and Boundary conditions** (e.g. concentration at $x=0$ and $x=1$).

If such information were included in the training process—for instance, if we knew that the concentration at both boundaries remains zero—the PINN performance could improve even further, and render it less dependent on extracting the information from the noisy measurement data. Ultimately, this allows is to train the PINN even **without any measurment data**.

*Let's test this in the next tutorial*.